# Máy chủ LLM local trên Colab — Qwen2.5-Coder-7B-Instruct

Notebook này biến một runtime GPU của Colab thành **máy chủ LLM tương thích OpenAI**
cho hệ chuyên gia xử phạt giao thông chạy ở máy bạn.

**Cách dùng**
1. `Runtime → Change runtime type → T4 GPU` (bản miễn phí là đủ).
2. Chạy lần lượt 4 ô bên dưới (ô nạp model mất ~5 phút lần đầu).
3. Ô cuối in ra một URL `https://….trycloudflare.com/v1`.
4. Mở Streamlit ở máy bạn → sidebar chọn **Local / Colab (Qwen…)** → dán URL vào ô
   *Base URL* → bấm **Thử kết nối**.

Giữ tab Colab mở trong lúc dùng: runtime miễn phí bị ngắt sau khoảng 90 phút không tương tác.


In [ ]:
# Ô 1 — cài thư viện và tải cloudflared (đường hầm công khai, không cần tài khoản)
!pip -q install "transformers>=4.45" accelerate bitsandbytes fastapi "uvicorn[standard]" nest_asyncio
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
print("Xong.")


In [ ]:
# Ô 2 — nạp model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

_TOK = None
_MODEL = None


def load_qwen(model_id: str = MODEL_ID, force_4bit=None):
    """Nạp model lên GPU, trả về (tokenizer, model); gọi lại thì dùng bản đã nạp.

    Colab miễn phí cấp T4 16GB, trong khi 7B ở fp16 đã tốn ~15GB nên không còn chỗ cho
    KV cache. Vì vậy mặc định lượng tử hóa 4-bit NF4 (~5.5GB) khi GPU dưới 24GB; máy
    A100/L4 thì để nguyên fp16 cho nhanh và chính xác hơn.
    """
    global _TOK, _MODEL
    if _MODEL is not None:
        return _TOK, _MODEL
    if not torch.cuda.is_available():
        raise RuntimeError("Chưa bật GPU: Runtime -> Change runtime type -> T4 GPU.")

    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    use_4bit = vram_gb < 24 if force_4bit is None else force_4bit
    print(f"GPU {torch.cuda.get_device_name(0)} · {vram_gb:.1f} GB · 4-bit={use_4bit}")

    kw = {"device_map": "auto", "torch_dtype": torch.float16}
    if use_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

    _TOK = AutoTokenizer.from_pretrained(model_id)
    _MODEL = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    _MODEL.eval()
    return _TOK, _MODEL


@torch.inference_mode()
def chat(messages, max_new_tokens: int = 1024, temperature: float = 0.0) -> str:
    """Sinh câu trả lời theo chat template của Qwen.

    temperature = 0 thì tắt sampling: hệ chuyên gia cần đầu ra JSON ổn định, lặp lại được.
    """
    tok, model = load_qwen()
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok([prompt], return_tensors="pt").to(model.device)
    sampling = {"temperature": temperature, "top_p": 0.9} if temperature > 0 else {}
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        pad_token_id=tok.eos_token_id,
        **sampling,
    )
    generated = out[0][inputs.input_ids.shape[-1] :]
    return tok.decode(generated, skip_special_tokens=True).strip()


load_qwen()
print(chat([{"role": "user", "content": 'Tra ve dung JSON: {"ok": true}'}]))


In [ ]:
# Ô 3 — dựng API tương thích OpenAI
import socket
import threading
import time
import uuid
from typing import List, Optional

import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

nest_asyncio.apply()

app = FastAPI(title="Qwen local server")
_GEN_LOCK = threading.Lock()  # một GPU, sinh tuần tự để không tranh bộ nhớ


class Message(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    messages: List[Message]
    model: Optional[str] = None
    temperature: float = 0.0
    max_tokens: int = 1024


@app.get("/v1/models")
def list_models():
    return {
        "object": "list",
        "data": [{"id": MODEL_ID, "object": "model", "owned_by": "local"}],
    }


@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    with _GEN_LOCK:
        text = chat(
            [m.model_dump() for m in req.messages],
            max_new_tokens=req.max_tokens,
            temperature=req.temperature,
        )
    return {
        "id": "chatcmpl-" + uuid.uuid4().hex[:12],
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model or MODEL_ID,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": text},
                "finish_reason": "stop",
            }
        ],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
    }


def port_open(port: int, host: str = "127.0.0.1") -> bool:
    with socket.socket() as sock:
        sock.settimeout(1.0)
        return sock.connect_ex((host, port)) == 0


def serve(port: int = 8000, timeout: int = 30):
    """Chạy uvicorn ở luồng nền, chờ tới khi cổng thật sự nhận kết nối.

    Chờ đúng trạng thái thay vì sleep một khoảng đoán trước: nếu uvicorn chết ngay
    (cổng bị chiếm, lỗi import) thì phải biết ngay ở đây, chứ để tới lúc cloudflared
    trả 502 Bad gateway thì rất khó đoán nguyên nhân.
    """
    config = uvicorn.Config(app, host="0.0.0.0", port=port, log_level="warning")
    thread = threading.Thread(target=uvicorn.Server(config).run, daemon=True)
    thread.start()
    deadline = time.time() + timeout
    while time.time() < deadline:
        if port_open(port):
            return thread
        time.sleep(0.5)
    raise RuntimeError(f"uvicorn không mở được cổng {port} sau {timeout}s.")


serve()
print("Server đang chạy ở http://localhost:8000")


In [ ]:
# Ô 4 — mở đường hầm và lấy URL công khai
import re
import subprocess
import time


def start_tunnel(port: int = 8000, timeout: int = 90):
    """Chạy cloudflared và trả về (url_công_khai, tiến_trình)."""
    # cloudflared vẫn cấp URL dù origin chưa sống, nên không kiểm ở đây thì lỗi chỉ
    # lộ ra ở máy người dùng dưới dạng "502 Bad gateway" — rất khó truy nguyên.
    if not port_open(port):
        raise RuntimeError(
            f"Chưa có server nào lắng nghe cổng {port}. Chạy xong ô 2 và ô 3 trước "
            "(ô 3 phải in được dòng 'Server đang chạy'), rồi mới chạy ô này."
        )
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    deadline = time.time() + timeout
    for line in proc.stdout:
        found = re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
        if found:
            return found.group(0), proc
        if time.time() > deadline:
            break
    proc.terminate()
    raise RuntimeError("Không lấy được URL từ cloudflared, thử chạy lại ô này.")


url, _tunnel = start_tunnel()
print("Dán URL sau vào ô 'Base URL của server LLM' trong Streamlit:\n")
print(f"    {url}/v1\n")
print("Kiểm tra nhanh:")
print(f"    curl {url}/v1/models")


## Dùng ở máy bạn

Chọn trên sidebar Streamlit, hoặc đặt sẵn trong `.env` để chế độ *Tự động* nhận ra:

```bash
TRAFFIC_ES_LLM_PROVIDER=local
TRAFFIC_ES_LOCAL_BASE_URL=https://xxx.trycloudflare.com/v1
TRAFFIC_ES_LOCAL_MODEL=Qwen/Qwen2.5-Coder-7B-Instruct
```

URL `trycloudflare.com` đổi mỗi lần chạy lại ô 4, nhớ dán lại URL mới.

**Lưu ý:** đường hầm này công khai với bất kỳ ai biết URL. Chỉ dùng để thử nghiệm, và
đóng runtime khi xong.

## Gặp lỗi 502 Bad gateway?

502 nghĩa là đường hầm sống nhưng bên trong Colab không có gì lắng nghe cổng 8000 —
gần như luôn là do ô 3 chưa chạy xong. Chạy ô sau để soi:

```python
print("cổng 8000 mở?", port_open(8000))
!curl -s -o /dev/null -w "origin trả HTTP %{http_code}\n" http://localhost:8000/v1/models
```

Nếu cổng chưa mở: chạy lại ô 2 (chờ nó in ra JSON thử), rồi ô 3 (chờ dòng
`Server đang chạy`), rồi mới tới ô 4.

Nếu origin trả 200 mà từ máy nhà vẫn 502 thì đường hầm đã đứt — chạy lại ô 4 để lấy URL mới.
